# Student Exam Performance( subset selection - backward elimination)

## 📦 Features we will create

### 📚 Study-related features (important signals)

- study_hours_per_day (strong positive)
- attendance_percentage (strong positive)
- previous_score (medium positive)

### 🧠 Lifestyle features

- sleep_hours (moderate positive)
- social_media_hours (negative effect)

### 🏫 Context features

- tutoring (0/1)
- family_income_level (low/medium/high encoded)

### 🎲 Noise feature

- random_noise (should be removed by backward elimination)

## Step 1: Create dataset

In [16]:
import pandas as pd
import numpy as np

# -----------------------------
# Settings
# -----------------------------
n = 2000
np.random.seed(42)

# -----------------------------
# Create raw dataset
# -----------------------------
df = pd.DataFrame({
    'study_hours_per_day': np.random.randint(0, 6, n),
    'attendance_percentage': np.random.randint(30, 85, n),
    'previous_score': np.random.randint(33, 96, n),
    'sleep_hours': np.random.randint(2, 8, n),
    'social_media_hours': np.random.randint(2, 14, n),

    'tutoring': np.random.randint(0, 2, n),

    # categorical feature (real-world style labels)
    'family_income_level': np.random.choice(['low', 'medium', 'high'], n),

    'random_noise': np.random.randn(n)
})

# -----------------------------
# Target variable (exam_score)
# -----------------------------
df['exam_score'] = (
    5 * df['study_hours_per_day']
    + 0.3 * df['attendance_percentage']
    + 0.4 * df['previous_score']
    + 2 * df['sleep_hours']
    - 3 * df['social_media_hours']

    + 5 * df['tutoring']

    # income effects (encoded logically BEFORE encoding step)
    + 0 * (df['family_income_level'] == 'low')
    + 4 * (df['family_income_level'] == 'medium').astype(int)
    + 7 * (df['family_income_level'] == 'high').astype(int)

    + np.random.randn(n) * 5
)

# -----------------------------
# One-hot encoding (for ML model)
# -----------------------------
df_encoded = pd.get_dummies(
    df,
    columns=['tutoring', 'family_income_level'],
    drop_first=True
)

# -----------------------------
# Check final columns
# -----------------------------
print("Final columns after encoding:\n")
print(df_encoded.columns.tolist())


# -----------------------------
# Preview dataset
# -----------------------------
print("\nFirst 5 rows:\n")
df_encoded.head()



Final columns after encoding:

['study_hours_per_day', 'attendance_percentage', 'previous_score', 'sleep_hours', 'social_media_hours', 'random_noise', 'exam_score', 'tutoring_1', 'family_income_level_low', 'family_income_level_medium']

First 5 rows:



,study_hours_per_day,attendance_percentage,previous_score,sleep_hours,social_media_hours,random_noise,exam_score,tutoring_1,family_income_level_low,family_income_level_medium
0,3,48,76,2,2,-1.755438,76.986053,True,False,True
1,4,84,76,4,5,-0.695149,74.335384,True,True,False
2,2,75,84,4,7,0.473145,59.652677,False,False,True
3,4,65,64,5,8,-0.702911,61.108693,True,False,True
4,4,74,38,6,11,-0.237807,48.051846,True,False,True


### Step 2: train and test split

In [17]:
from sklearn.model_selection import train_test_split

X = df_encoded.drop(columns = "exam_score")
y = df["exam_score"]

X_train, X_test, y_train, y_test = train_test_split( X, y, test_size= 0.2, random_state= 42)

### Step 3: Evaluation function

In [19]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

def evaluate(features):

    model = LinearRegression()
    
    # Train only on selected features
    model.fit(X_train[features], y_train)

    # predict on test set
    prediction=  model.predict(X_test[features])

    #return MSE on test set
    return mean_squared_error(y_test, prediction)

### Step 4: Backward Elimination

In [22]:
selected = list(X_train.columns)
prev_score = evaluate(selected)

while len(selected) > 1:

    scores = []

    for f in selected:
        temp_features = selected.copy()
        temp_features.remove(f)
        score = evaluate(temp_features)
        scores.append((score, f))

    scores.sort()
    best_score, feature_to_remove = scores[0]

    if best_score > prev_score:
        break

    selected.remove(feature_to_remove)
    prev_score = best_score

    print(f"Removed: {feature_to_remove}, MSE: {best_score:.2f}")
    
print("\nFinal selected features:")
print(selected)

Removed: random_noise, MSE: 22.11

Final selected features:
['study_hours_per_day', 'attendance_percentage', 'previous_score', 'sleep_hours', 'social_media_hours', 'tutoring_1', 'family_income_level_low', 'family_income_level_medium']
